<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Python%20Fundamentals%20for%20AI%20Apps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐍 Python Fundamentals for Building GenAI Apps

Every AI app in this bootcamp — chatbots, RAG pipelines, tool-calling agents — is just **Python moving text and dictionaries around an API call**. This notebook is the missing "Day 0": the Python building blocks you'll actually lean on, each one taught next to the exact GenAI job it does.

No API key, no LM Studio, no internet required — every cell below runs anywhere Python runs. Where a concept maps directly onto something you'll see later in the bootcamp (a chat `messages` list, a tool schema, a streaming loop), it's called out explicitly.

**You'll learn:** variables & types → strings/f-strings (prompt building) → lists/dicts (chat history & JSON) → control flow & comprehensions → functions & type hints (tool calling) → JSON parsing (API responses) → error handling (flaky APIs) → files (loading documents for RAG) → classes (building a Chatbot/Agent) → generators (token streaming) → decorators (retries) → environment variables (API keys) → a final mini-project that wires all of it into a tiny mock "AI agent".

## ✅ Prerequisites

- Python 3.9+ (any install — no `venv` activation strictly required for this notebook, since nothing outside the standard library is used until the very end)
- Nothing else. This notebook is intentionally offline and dependency-free.

## 📖 1. Variables & Data Types

A **variable** is just a name pointing at a value. Python figures out the type for you (int, float, str, bool, `None`) — you never declare it up front. This matters in GenAI code because API responses mix these types freely (a `temperature` is a float, a `model` name is a string, `stream` is a bool) and Python won't stop you from mixing them up — so knowing the type of what you're holding is on you.

In [1]:
model_name = "gpt-4o-mini"       # str
temperature = 0.7                 # float
max_tokens = 500                  # int
stream_response = True            # bool
system_prompt = None              # NoneType — "not set yet"

for name, value in [
    ("model_name", model_name),
    ("temperature", temperature),
    ("max_tokens", max_tokens),
    ("stream_response", stream_response),
    ("system_prompt", system_prompt),
]:
    print(f"{name:16s} = {value!r:20s}  ({type(value).__name__})")

model_name       = 'gpt-4o-mini'         (str)
temperature      = 0.7                   (float)
max_tokens       = 500                   (int)
stream_response  = True                  (bool)
system_prompt    = None                  (NoneType)


## 📖 2. Strings & f-strings — the real "prompt engineering" tool

A prompt is nothing but a string. **f-strings** (`f"...{variable}..."`) are how you'll build prompts from variables in almost every notebook in this bootcamp — cleaner and safer than string concatenation with `+`.

In [2]:
user_name = "Priya"
topic = "black holes"
reading_level = "a curious 10-year-old"

# ❌ concatenation — easy to get spacing/types wrong, painful to read
prompt_v1 = "Explain " + topic + " to " + user_name + " at the level of " + reading_level + "."

# ✅ f-string — variables inline, easy to read, easy to extend
prompt_v2 = f"Explain {topic} to {user_name} at the level of {reading_level}."

print(prompt_v1)
print(prompt_v2)

Explain black holes to Priya at the level of a curious 10-year-old.
Explain black holes to Priya at the level of a curious 10-year-old.


In [3]:
# Multi-line prompts (system instructions, few-shot examples) use triple-quoted strings.
system_prompt = f"""You are a helpful tutor.
Student name: {user_name}
Explain topics simply, in under 3 sentences.
"""
print(system_prompt)

# .strip() and .format() are the other two string tools you'll see constantly:
messy = "   some API responses have trailing whitespace\n\n"
print(repr(messy.strip()))

template = "Score: {score}/100 — {verdict}"
print(template.format(score=87, verdict="pass"))

You are a helpful tutor.
Student name: Priya
Explain topics simply, in under 3 sentences.

'some API responses have trailing whitespace'
Score: 87/100 — pass


## 📖 3. Lists, Tuples, Sets & Dictionaries

These four containers are the backbone of every GenAI app:

| Container | Ordered? | Mutable? | Duplicates? | GenAI use |
|---|---|---|---|---|
| `list` `[...]` | ✅ | ✅ | ✅ | Chat history, retrieved chunks, batches |
| `tuple` `(...)` | ✅ | ❌ | ✅ | Fixed pairs, e.g. `(role, content)` |
| `set` `{...}` | ❌ | ✅ | ❌ | Deduplicating retrieved doc IDs |
| `dict` `{k: v}` | ✅ (3.7+) | ✅ | keys unique | **The shape of every API request & response** |

The single most important pattern in this table: an LLM chat request is *a list of dicts*. Every provider (OpenAI, Anthropic, Gemini, LM Studio) uses this exact shape.

In [4]:
# This IS the actual shape sent to a chat completion API.
conversation = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "What's the capital of France?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user", "content": "And Japan?"},
]

for turn in conversation:
    print(f"{turn['role']:9s}: {turn['content']}")

print("\nTurns so far:", len(conversation))
print("Just the user messages:", [t["content"] for t in conversation if t["role"] == "user"])

system   : You are a concise assistant.
user     : What's the capital of France?
assistant: Paris.
user     : And Japan?

Turns so far: 4
Just the user messages: ["What's the capital of France?", 'And Japan?']


In [5]:
# Sets: deduplicate retrieved document IDs after a RAG search (duplicates are common
# when multiple chunks come from the same source document).
retrieved_doc_ids = ["doc_1", "doc_3", "doc_1", "doc_7", "doc_3"]
unique_doc_ids = set(retrieved_doc_ids)
print("Retrieved:", retrieved_doc_ids)
print("Unique sources to cite:", unique_doc_ids)

# Tuples: an immutable (role, content) pair — safe to use as a dict key or in a set,
# because tuples can't be accidentally mutated later.
pair = ("user", "Hello!")
print("Immutable pair:", pair)

Retrieved: ['doc_1', 'doc_3', 'doc_1', 'doc_7', 'doc_3']
Unique sources to cite: {'doc_3', 'doc_1', 'doc_7'}
Immutable pair: ('user', 'Hello!')


## 📖 4. Control Flow & Comprehensions

`if`/`elif`/`else`, `for`, and `while` work exactly like you'd expect. The one Python-specific superpower is the **comprehension** — a one-line loop that builds a new list/dict/set. You'll use these constantly to reshape API data: filtering low-confidence results, extracting one field from a list of dicts, or splitting a document into chunks.

In [6]:
scores = [0.92, 0.41, 0.78, 0.15, 0.88]

# Classic loop
above_threshold = []
for s in scores:
    if s >= 0.7:
        above_threshold.append(s)
print("Loop version:      ", above_threshold)

# Same result, one line — a list comprehension
above_threshold = [s for s in scores if s >= 0.7]
print("Comprehension:      ", above_threshold)

# Dict comprehension: pair each score with a pass/fail verdict
verdicts = {s: ("pass" if s >= 0.7 else "fail") for s in scores}
print("Dict comprehension: ", verdicts)

Loop version:       [0.92, 0.78, 0.88]
Comprehension:       [0.92, 0.78, 0.88]
Dict comprehension:  {0.92: 'pass', 0.41: 'fail', 0.78: 'pass', 0.15: 'fail', 0.88: 'pass'}


In [7]:
# Realistic RAG example: chunk a long document into fixed-size pieces using a
# `while` loop, then use a comprehension to keep only non-empty chunks.
document = ("Retrieval-Augmented Generation lets a model read your own documents. " * 4).strip()
chunk_size = 60

chunks = []
start = 0
while start < len(document):
    chunks.append(document[start:start + chunk_size])
    start += chunk_size

non_empty_chunks = [c for c in chunks if c.strip()]
print(f"Document length: {len(document)} chars -> {len(non_empty_chunks)} chunks")
for i, c in enumerate(non_empty_chunks[:3]):
    print(f"  chunk {i}: {c!r}")

Document length: 275 chars -> 5 chunks
  chunk 0: 'Retrieval-Augmented Generation lets a model read your own do'
  chunk 1: 'cuments. Retrieval-Augmented Generation lets a model read yo'
  chunk 2: 'ur own documents. Retrieval-Augmented Generation lets a mode'


## 📖 5. Functions, `*args`/`**kwargs` & Type Hints

Every SDK call you'll write (`client.chat.completions.create(...)`) is a function call with **keyword arguments**. And every "tool calling" / "function calling" feature in agent frameworks works by reading a Python function's **name, parameters, type hints, and docstring** to build a schema the model can call. Writing clean functions now is directly writing tool definitions later.

In [8]:
def build_prompt(topic: str, audience: str = "a beginner", max_sentences: int = 3) -> str:
    """Build a short explanation prompt.

    Args:
        topic: What to explain.
        audience: Who the explanation is for.
        max_sentences: Hard cap on response length.
    """
    return f"Explain {topic} to {audience} in under {max_sentences} sentences."

print(build_prompt("neural networks"))
print(build_prompt("neural networks", audience="a software engineer", max_sentences=1))

# **kwargs mirrors how you pass options straight through to an API client
def call_model(prompt: str, **options) -> None:
    print(f"prompt={prompt!r}")
    print(f"options={options}")

call_model("Hello!", temperature=0.2, max_tokens=100, model="gpt-4o-mini")

Explain neural networks to a beginner in under 3 sentences.
Explain neural networks to a software engineer in under 1 sentences.
prompt='Hello!'
options={'temperature': 0.2, 'max_tokens': 100, 'model': 'gpt-4o-mini'}


In [9]:
# This is the function-calling / tool-use pattern in miniature: a plain Python
# function with type hints becomes a "tool" the model can be told about and asked to call.
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Look up the current weather for a city. (Fake data — no network call.)"""
    fake_db = {"Paris": 18, "Tokyo": 24, "Mumbai": 31}
    temp = fake_db.get(city, 20)
    return {"city": city, "temperature": temp, "unit": unit}

# An agent framework would inspect get_weather.__name__, its type hints, and its
# docstring to build a JSON schema describing this tool to the model. We can do
# the "reading the signature" part ourselves with the standard library:
import inspect
sig = inspect.signature(get_weather)
print("Tool name:", get_weather.__name__)
print("Tool description:", get_weather.__doc__.strip())
print("Parameters:")
for name, param in sig.parameters.items():
    print(f"  - {name}: {param.annotation.__name__ if param.annotation != inspect._empty else 'any'}")

print("\nCalling it:", get_weather("Tokyo"))

Tool name: get_weather
Tool description: Look up the current weather for a city. (Fake data — no network call.)
Parameters:
  - city: str
  - unit: str

Calling it: {'city': 'Tokyo', 'temperature': 24, 'unit': 'celsius'}


## 📖 6. JSON — the language every LLM API speaks

HTTP APIs (including every LLM provider) send and receive **JSON text**, which Python's `json` module converts to/from dicts and lists. `json.dumps` = Python → JSON text (for a request body). `json.loads` = JSON text → Python (for a response). You'll also frequently ask a model to *reply in JSON* so you can parse its answer programmatically instead of scraping free text.

In [10]:
import json

# Python dict -> JSON string, exactly what gets sent over the wire in an API request
request_body = {
    "model": "gpt-4o-mini",
    "messages": [{"role": "user", "content": "Say hi in one word."}],
    "temperature": 0,
}
request_json = json.dumps(request_body, indent=2)
print(request_json)
print(type(request_json))

{
  "model": "gpt-4o-mini",
  "messages": [
    {
      "role": "user",
      "content": "Say hi in one word."
    }
  ],
  "temperature": 0
}
<class 'str'>


In [11]:
# JSON string -> Python dict, exactly what happens when you parse an API's response
response_json = """
{
  "id": "chatcmpl-123",
  "choices": [
    {"message": {"role": "assistant", "content": "Hi!"}, "finish_reason": "stop"}
  ],
  "usage": {"prompt_tokens": 12, "completion_tokens": 2}
}
"""
response = json.loads(response_json)
print(type(response))
print("Assistant said:", response["choices"][0]["message"]["content"])
print("Tokens used:", response["usage"]["prompt_tokens"] + response["usage"]["completion_tokens"])

<class 'dict'>
Assistant said: Hi!
Tokens used: 14


In [12]:
# A model asked to "reply only in JSON" gives you a string you must parse yourself —
# and that string is sometimes malformed. Always wrap the parse in a try/except (next section).
model_reply_text = '{"sentiment": "positive", "confidence": 0.94}'

try:
    parsed = json.loads(model_reply_text)
    print("Parsed OK:", parsed)
    print("Sentiment:", parsed["sentiment"])
except json.JSONDecodeError as e:
    print("Model did not return valid JSON:", e)

Parsed OK: {'sentiment': 'positive', 'confidence': 0.94}
Sentiment: positive


## 📖 7. Error Handling — because APIs *will* fail

Network calls to an LLM provider can time out, hit a rate limit, or return malformed data. `try`/`except`/`finally` is how you keep one bad response from crashing the whole app, and it's the foundation of **retry logic**, which every production AI app needs.

In [13]:
import random

class RateLimitError(Exception):
    """Custom exception — clearer than a generic Exception when reading a traceback."""
    pass

def flaky_api_call(fail_chance: float = 0.7) -> str:
    """Simulates an LLM API call that sometimes gets rate-limited."""
    if random.random() < fail_chance:
        raise RateLimitError("429: Too Many Requests")
    return "Here is your AI-generated answer."

random.seed(7)  # deterministic output for this demo

try:
    result = flaky_api_call()
    print("Success:", result)
except RateLimitError as e:
    print("Handled gracefully:", e)
finally:
    print("This always runs — great place to log or close a connection.")

Handled gracefully: 429: Too Many Requests
This always runs — great place to log or close a connection.


In [14]:
# The real pattern: retry a few times with backoff before giving up.
import time

def call_with_retry(fn, max_attempts: int = 5) -> str:
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(fail_chance=0.6)
        except RateLimitError as e:
            wait = 0.01 * attempt  # kept tiny so the notebook runs fast
            print(f"Attempt {attempt} failed ({e}) — retrying in {wait:.2f}s...")
            time.sleep(wait)
    raise RuntimeError(f"Gave up after {max_attempts} attempts")

random.seed(3)
print("\nFinal result:", call_with_retry(flaky_api_call))

Attempt 1 failed (429: Too Many Requests) — retrying in 0.01s...
Attempt 2 failed (429: Too Many Requests) — retrying in 0.02s...


Attempt 3 failed (429: Too Many Requests) — retrying in 0.03s...



Final result: Here is your AI-generated answer.


## 📖 8. Files & Context Managers — loading documents for RAG

`with open(...) as f:` is a **context manager**: it guarantees the file gets closed even if an error happens inside the block. This is exactly how a RAG pipeline loads source documents from disk before chunking and embedding them.

In [15]:
from pathlib import Path

# Write a small sample "knowledge base" file, then read it back — the same
# read step a RAG pipeline runs before chunking + embedding.
sample_path = Path("sample_knowledge.txt")
sample_path.write_text(
    "Retrieval-Augmented Generation (RAG) combines a retriever with a language model.\n"
    "The retriever finds relevant text chunks; the model uses them as context to answer.\n",
    encoding="utf-8",
)

with open(sample_path, "r", encoding="utf-8") as f:
    contents = f.read()

print(contents)
print("Lines:", len(contents.splitlines()))

sample_path.unlink()  # clean up the demo file
print("Cleaned up:", not sample_path.exists())

Retrieval-Augmented Generation (RAG) combines a retriever with a language model.
The retriever finds relevant text chunks; the model uses them as context to answer.



Lines: 2
Cleaned up: True


## 📖 9. Classes & OOP — packaging a chatbot's state

A chatbot needs to *remember* its conversation between calls. A plain function can't hold state between calls the way an **object** can — a class bundles data (`self.history`) with the behavior that acts on it (`send`), which is exactly how agent/chatbot frameworks are structured under the hood.

In [16]:
class Chatbot:
    """A minimal chatbot that keeps its own conversation history."""

    def __init__(self, system_prompt: str):
        self.history = [{"role": "system", "content": system_prompt}]

    def send(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        reply = self._fake_llm_call()
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def _fake_llm_call(self) -> str:
        """Stands in for a real API call — echoes context length instead."""
        last_user_msg = self.history[-1]["content"]
        return f"(mock reply to: {last_user_msg!r}) — I now recall {len(self.history)} messages."

bot = Chatbot(system_prompt="You are a terse assistant.")
print(bot.send("Hi there!"))
print(bot.send("What's 2 + 2?"))

print("\nFull history object:")
for turn in bot.history:
    print(" ", turn)

(mock reply to: 'Hi there!') — I now recall 2 messages.
(mock reply to: "What's 2 + 2?") — I now recall 4 messages.

Full history object:
  {'role': 'system', 'content': 'You are a terse assistant.'}
  {'role': 'user', 'content': 'Hi there!'}
  {'role': 'assistant', 'content': "(mock reply to: 'Hi there!') — I now recall 2 messages."}
  {'role': 'user', 'content': "What's 2 + 2?"}
  {'role': 'assistant', 'content': '(mock reply to: "What\'s 2 + 2?") — I now recall 4 messages.'}


## 📖 10. Iterators & Generators — how token streaming works

When a chat UI prints a response word-by-word instead of waiting for the whole thing, that's a **generator**: a function using `yield` instead of `return`, which produces values one at a time instead of building the whole list in memory first. This is *exactly* how `stream=True` works in every LLM SDK.

In [17]:
import time

def fake_token_stream(text: str):
    """Yields one word at a time with a tiny delay — a stand-in for a real streaming API."""
    for word in text.split():
        time.sleep(0.02)
        yield word + " "

print("Streaming response:")
for token in fake_token_stream("This is what a streamed AI response looks like token by token"):
    print(token, end="", flush=True)
print("\n\nDone.")

Streaming response:
This 

is 

what 

a 

streamed 

...

like 

token 

by 

token 



Done.


## 📖 11. Decorators — wrapping every API call with retry/timing logic

A **decorator** (`@something`) wraps a function to add behavior *around* it — logging, timing, or retries — without touching the function's own code. Agent frameworks decorate tool functions constantly; here's the pattern from scratch.

In [18]:
import functools
import time

def timed(fn):
    """Decorator: prints how long the wrapped function took to run."""
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[{fn.__name__}] took {elapsed*1000:.1f} ms")
        return result
    return wrapper

@timed
def simulate_llm_call(prompt: str) -> str:
    time.sleep(0.05)  # pretend this is network latency
    return f"Answer to: {prompt}"

print(simulate_llm_call("What is Python?"))

[simulate_llm_call] took 60.4 ms


Answer to: What is Python?


## 📖 12. Environment Variables — never hardcode an API key

Hardcoding `api_key = "sk-..."` into a notebook means it can leak the moment you share the file or push to GitHub. The standard fix: read secrets from **environment variables** (often loaded from a local `.env` file via `python-dotenv`, as this repo's project templates do) so the key never appears in your source code.

In [19]:
import os

# In a real project: `pip install python-dotenv`, put OPENAI_API_KEY=... in a
# gitignored .env file, then call load_dotenv() once at startup:
#
#   from dotenv import load_dotenv
#   load_dotenv()
#
# Here we simulate that step by setting the variable directly, to show the read side.
os.environ["DEMO_API_KEY"] = "sk-demo-not-a-real-key"

api_key = os.environ.get("DEMO_API_KEY")
missing_key = os.environ.get("SOME_KEY_THAT_ISNT_SET")  # .get() returns None instead of crashing

print("Loaded key (masked):", api_key[:6] + "..." if api_key else None)
print("Missing key falls back to:", missing_key)

if not missing_key:
    print("\n✅ This is the check every app should run at startup before calling an API:")
    print('   if not os.environ.get("OPENAI_API_KEY"): raise RuntimeError("Missing API key")')

Loaded key (masked): sk-dem...
Missing key falls back to: None

✅ This is the check every app should run at startup before calling an API:
   if not os.environ.get("OPENAI_API_KEY"): raise RuntimeError("Missing API key")


## 🧪 Putting it all together — a tiny mock "AI agent"

One last demo that chains everything above into a single, runnable loop: a **class** holds state, a **function with type hints** acts as a callable "tool", **JSON** carries the tool call, a **dict-based history** tracks the conversation, and a **try/except** protects against a bad tool call. No API key needed — the "model" is a tiny rule-based stand-in, but the *shape* of this loop is identical to a real tool-calling agent.

In [20]:
import json

def get_weather(city: str) -> dict:
    """Tool: look up fake weather data for a city."""
    fake_db = {"Paris": 18, "Tokyo": 24, "Mumbai": 31}
    if city not in fake_db:
        raise ValueError(f"No weather data for {city!r}")
    return {"city": city, "temperature_c": fake_db[city]}

TOOLS = {"get_weather": get_weather}

class MiniAgent:
    """A minimal tool-calling agent loop, built entirely from the concepts above."""

    def __init__(self):
        self.history = []

    def _fake_model_decides_tool_call(self, user_message: str) -> dict:
        """Stands in for an LLM deciding to call a tool. Real models return this
        same {"tool": ..., "arguments": {...}} shape as structured JSON."""
        for city in ["Paris", "Tokyo", "Mumbai", "Berlin"]:
            if city.lower() in user_message.lower():
                return {"tool": "get_weather", "arguments": {"city": city}}
        return {"tool": None}

    def ask(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        decision = self._fake_model_decides_tool_call(user_message)
        print("Model's tool-call decision:", json.dumps(decision))

        if decision["tool"] is None:
            reply = "I don't have a tool for that."
        else:
            tool_fn = TOOLS[decision["tool"]]
            try:
                result = tool_fn(**decision["arguments"])
                reply = f"It's {result['temperature_c']}°C in {result['city']}."
            except ValueError as e:
                reply = f"Tool error: {e}"

        self.history.append({"role": "assistant", "content": reply})
        return reply

agent = MiniAgent()
for question in ["What's the weather in Tokyo?", "How about Berlin?", "Tell me a joke."]:
    print(f"\nUser: {question}")
    print("Agent:", agent.ask(question))

print(f"\nConversation now has {len(agent.history)} turns in memory.")


User: What's the weather in Tokyo?
Model's tool-call decision: {"tool": "get_weather", "arguments": {"city": "Tokyo"}}
Agent: It's 24°C in Tokyo.

User: How about Berlin?
Model's tool-call decision: {"tool": "get_weather", "arguments": {"city": "Berlin"}}
Agent: Tool error: No weather data for 'Berlin'

User: Tell me a joke.
Model's tool-call decision: {"tool": null}
Agent: I don't have a tool for that.

Conversation now has 6 turns in memory.


## 📝 Recap

| Concept | GenAI use you'll see it in |
|---|---|
| Variables & types | Config values (`temperature`, `max_tokens`, `model`) |
| f-strings | Building prompts from variables and templates |
| Lists of dicts | The `messages` array sent to every chat API |
| Sets / tuples | Deduplicating retrieved sources; immutable pairs |
| Comprehensions | Filtering results, chunking documents |
| Functions + type hints | Tool/function-calling schemas for agents |
| JSON (`dumps`/`loads`) | Every API request body and response |
| `try`/`except` | Handling rate limits, timeouts, malformed model output |
| `with open(...)` | Loading source documents for RAG |
| Classes | Chatbot/agent objects that hold conversation state |
| Generators (`yield`) | Streaming a response token-by-token |
| Decorators | Wrapping API calls with timing/retry logic |
| Environment variables | Keeping API keys out of source code |

**Next:** [`Day 1 - LLM Fundamentals/Learning`](../Day%201%20-%20LLM%20Fundamentals/Learning) — put these building blocks to work against a real model.

## 🏋️ Try It Yourself (optional)

Using only what's defined above (or the standard library):

1. Write a function `summarize_history(history: list) -> dict` that takes a `messages`-style list of dicts and returns `{"user_turns": N, "assistant_turns": N}`.
2. Add a second tool (e.g. `convert_currency(amount: float, to: str) -> dict`) to `TOOLS` in the mini-agent, and extend `_fake_model_decides_tool_call` to route to it.
3. Turn `fake_token_stream` into a version that yields whole sentences instead of words, using `text.split(". ")`.
4. Wrap `get_weather` with the `@timed` decorator from Section 11 and call it a few times.